In [1]:
!pip install -q pykan

from google.colab import drive
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import gc

# Monta Google Drive
drive.mount('/content/drive')

# Caricamento del file CSV da Drive
file_path = '/content/drive/MyDrive/Tesi_CICIoT/train.csv'

print("Caricamento del dataset in corso...")
df = pd.read_csv(file_path)

# Creazione etichetta binaria
label_col = 'label' if 'label' in df.columns else 'Label'
is_benign = df[label_col].str.lower().str.contains('benign')
df['binary_label'] = (~is_benign).astype(int)

print(f"Totale righe caricate: {len(df)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Caricamento del dataset in corso...
Totale righe caricate: 5491971
Numero di classi uniche rilevate: 34


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# 1. Encoding 33 classi
le = LabelEncoder()
df['multi_label'] = le.fit_transform(df[label_col])
y_raw = df['multi_label'].values

# 2. Split Stratificato sui dati GREZZI su tutte le 33 classi
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# 3. Scaling rigido senza Data Leakage
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# 4. Calcolo pesi per la CrossEntropyLoss
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# Loss Pesata Multiclasse
criterion = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

# ... [Ciclo di training standard] ...

# Valutazione
model.eval()
with torch.no_grad():
    test_outputs = model(dataset['test_input'])
    preds = torch.argmax(test_outputs, dim=1).cpu().numpy()
    labels = dataset['test_label'].cpu().numpy()

# Visualizzazione Matrice di Confusione (Normalizzata su Recall)
cm = confusion_matrix(labels, preds, normalize='true')
plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predizioni')
plt.ylabel('Classi Reali')
plt.title('Matrice di Confusione Multiclasse KAN')
plt.show()